# Yatharth Music AI — Free GPU real-AI test v3

Headless ACE-Step startup gate, real engine readiness, then temporary HTTPS testing. Colab remains a test environment, not 24/7 hosting.

In [ ]:
import os
os.environ['MPLBACKEND']='Agg'
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
!nvidia-smi || true
!rm -rf /content/ACE-Step-1.5 /content/yatharth-music-ai
!git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git /content/ACE-Step-1.5
!git clone --depth 1 https://github.com/rampaulsaini/yatharth-music-ai.git /content/yatharth-music-ai
%cd /content/ACE-Step-1.5
!pip -q install uv
!uv sync --frozen
%cd /content/yatharth-music-ai
!pip -q install -r requirements.txt
print('Headless backend:',os.environ['MPLBACKEND'])

In [ ]:
import subprocess,time,requests,os
os.environ['MPLBACKEND']='Agg'
log=open('/content/acestep.log','a')
cmd=['uv','run','python','-m','acestep.api_server','--host','127.0.0.1','--port','8001']
proc=None; ready=False
for attempt in range(1,4):
    if proc is None or proc.poll() is not None:
        print('ACE-Step startup attempt',attempt)
        proc=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,cwd='/content/ACE-Step-1.5',env=os.environ.copy())
    for _ in range(45):
        time.sleep(4)
        if proc.poll() is not None: break
        try:
            r=requests.get('http://127.0.0.1:8001/health',timeout=5)
            if r.ok: print(r.status_code,r.text[:500]); ready=True; break
        except requests.RequestException: pass
    if ready: break
if not ready:
    print(open('/content/acestep.log',errors='ignore').read()[-10000:])
    raise RuntimeError('ACE-Step did not become healthy after bounded retries.')
print('ACE-Step READY:',proc.pid)

In [ ]:
import subprocess,time,requests,os
ylog=open('/content/yatharth.log','w'); env=os.environ.copy()
env.update(DEMO_MODE='false',MUSIC_ENGINE_URL='http://127.0.0.1:8001',TRUST_PROXY='false',MPLBACKEND='Agg')
backend=subprocess.Popen(['python','-m','uvicorn','main:app','--host','0.0.0.0','--port','8000','--proxy-headers','--forwarded-allow-ips','127.0.0.1'],stdout=ylog,stderr=subprocess.STDOUT,cwd='/content/yatharth-music-ai',env=env)
ready=False
for _ in range(20):
    time.sleep(2)
    try:
        r=requests.get('http://127.0.0.1:8000/api/health',timeout=10); body=r.json(); print(body)
        if r.ok and body.get('ok') and body.get('engine_reachable'): ready=True; break
    except requests.RequestException: pass
if not ready:
    print(open('/content/yatharth.log',errors='ignore').read()[-8000:]); raise RuntimeError('Yatharth/ACE-Step readiness gate failed.')
print('Yatharth READY:',backend.pid)

In [ ]:
import subprocess,re,time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
log=open('/content/cloudflared.log','w')
tunnel=subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'],stdout=log,stderr=subprocess.STDOUT)
url=None
for _ in range(30):
    time.sleep(2); text=open('/content/cloudflared.log',errors='ignore').read(); m=re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com',text)
    if m: url=m.group(0); break
if not url: raise RuntimeError('Temporary public URL was not created.')
print('YATHARTH PUBLIC LINK:',url)
print('Keep this Colab runtime alive while testing.')

## Verification

Generate a short 10–30 second song. Real success requires phone → Yatharth → FastAPI → healthy ACE-Step → actual audio. A demo tone does not count.

For permanent 24/7 service, migrate to persistent GPU workers + durable queue/state + object storage + health replacement + failover.

In [ ]:
print(open('/content/acestep.log',errors='ignore').read()[-8000:])
print(open('/content/yatharth.log',errors='ignore').read()[-8000:])
print(open('/content/cloudflared.log',errors='ignore').read()[-5000:])